# Fraquezas de vetores e embeddings: a tese sob embargo na resposta errada

OWASP Top 10 para LLMs 2025 — **LLM08 Vector & Embedding Weaknesses (novo em 2025)**

Simulador didático (`campus/`), sem rede e sem chaves. Cenário do Campus Aurora (universidade fictícia, dados sintéticos).

In [1]:
import os, sys
_dir = os.path.abspath(os.getcwd())
for _ in range(5):
    if os.path.isdir(os.path.join(_dir, "campus")):
        break
    _dir = os.path.dirname(_dir)
sys.path.insert(0, _dir)

from campus import dados, defesas, ferramentas, narrar  # noqa: E402
from campus.llm import SimLLM  # noqa: E402

print("Aviso: SimLLM é um SIMULADOR didático, não um modelo real. Uso educacional.")

Aviso: SimLLM é um SIMULADOR didático, não um modelo real. Uso educacional.


## Cenário
O Acervo Aurora indexa uma tese sob embargo de patente do PPG-Saúde. Um aluno do PPG-Computação pergunta sobre o tema e a busca por semelhança devolve o trecho embargado, porque mede proximidade de texto, não quem tem direito de ler.

In [2]:
ALUNO = "A-1001"  # Marina Tavares, PPG-Computação
PERGUNTA = "Existe estudo sobre biossensor de baixo custo para triagem? Qual o método?"
programa_aluno = dados.ALUNOS[ALUNO]["programa"]


## Sem defesa

In [3]:
indice_vulneravel = dados.Indice(dados.ACERVO)
achados = indice_vulneravel.buscar(PERGUNTA, k=1)
for d in achados:
    print(d["id"], "- embargada:", d["embargada"], "-", d["programa"])
    print(" ", d["texto"][:140], "...")
assert achados[0]["embargada"] and achados[0]["programa"] != programa_aluno


T-S-007 - embargada: True - PPG-Saúde
  Resumo: o biossensor de baixo custo para triagem usa uma liga sintética (pedido de patente fictício PAT-SIM-0042) e atinge sensibilidade ele ...


## Com defesa
`defesas.filtrar_por_programa` remove, antes da busca, qualquer documento embargado que não seja do próprio programa do aluno (documentos com instrução oculta embutida também iriam para `defesas.quarentena`, complementando esta defesa — não é o caso desta tese específica).

In [4]:
acervo_filtrado = defesas.filtrar_por_programa(dados.ACERVO, programa_aluno)
indice_defendido = dados.Indice(acervo_filtrado)
achados_defendidos = indice_defendido.buscar(PERGUNTA, k=1)
for d in achados_defendidos:
    print(d["id"], "-", d["programa"], "-", d["titulo"])
assert all(not d["embargada"] or d["programa"] == programa_aluno for d in achados_defendidos)


## O que observar
- As fraquezas típicas de vetores/embeddings: vazamento entre tenants (este caso), envenenamento do índice por quem pode inserir documentos, e inversão de embeddings (reconstituir texto a partir do vetor) — os dois últimos não aparecem neste notebook, mas usam o mesmo índice compartilhado como porta de entrada.
- A autorização precisa ser aplicada ANTES da busca; filtrar a resposta depois já é tarde, o trecho já foi lido pelo modelo.
- Trate vetores como dado sensível: quem pode indexar documentos é, na prática, quem decide o que pode vazar.